# WeedBot weed detector v2 - step 4: label the new photos, then train v2
First a big model (OWLv2, the "teacher") draws a box around the plant in each photo from step 3 (about 3-4.5
hours of T4 time). Then the small detector is trained again, starting from v1, on all the photos (about 4-5 hours).
The result goes to your Google Drive as `MyDrive/weedbot/weed_detector_v2.zip`.

**How:** Runtime -> Change runtime type -> **T4 GPU**, then Runtime -> **Run all**. Allow Google Drive access.
Keep the laptop awake. This needs more GPU time than free Colab gives in one day: plan for 2-3 days (the first
day is mostly labelling).
When Colab stops, come back later, open this notebook again and press **Run all**: finished labelling is kept
on Drive (it prints `already labelled on Drive, skipped`) and training continues from its last round. After each
restart it first unpacks and prepares the photos for about 10-20 minutes before training goes on: that is
normal, it is not stuck.
The install cell may print red warnings about other packages ("dependency conflicts"): expected, ignore them.
If Colab asks you to restart the session, click Restart session, then Runtime -> **Run all** again. If it stops
with `transformers 4.57.6 is not active`, do what it says (Runtime > Restart session, then Runtime > Run all);
if it stops there again, tell Claude.
If Colab says the GPU is not available, try again later (do not switch to CPU).
When it prints **ALL DONE**, download `weed_detector_v2.zip` from Drive (MyDrive/weedbot) and send it to Claude.
The first zip you send Claude is the final v2 model: do not run step 4 or 4b again after that.
If it stopped near the end (for example round 47 of 50) and Colab says "Cannot connect to GPU backend",
use step 4b (`04b_v2_finish_without_gpu.ipynb`) instead.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys
DRIVE = '/content/drive/MyDrive/weedbot'
os.makedirs(DRIVE, exist_ok=True)
sys.path.insert(0, '/content')

import torch
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> T4 GPU (if none is free, try later)'
print('GPU:', torch.cuda.get_device_name(0))

assert os.path.exists(DRIVE + '/runs/v1/weights/best.pt'), 'v1 must be trained first; keep the v1 files on Drive'

In [ ]:
import os
os.makedirs("/content/weed_detector", exist_ok=True)

In [ ]:
%%writefile /content/weed_detector/__init__.py
"""WeedBot weed/keep detector: dataset building (sources, build, colab) and training (train)."""

In [ ]:
%%writefile /content/weed_detector/sources.py
"""sources.py - open weed / plant datasets -> items for ONE two-class YOLO dataset (weed = 0, keep = 1).

Why: the WeedBot detector learns from public photos only (the owner takes no photos). Each source has its
own labels; this file maps them onto two classes. Design: docs/superpowers/specs/2026-11-24-weed-detector-design.md.

  cottonweed   CottonWeedDet12 via Hugging Face Voxel51/CottonWeedDet12 (samples.json, FiftyOne boxes):
               all 12 weed species -> weed, EXCEPT grass-type weeds (Goosegrass): their boxes are dropped,
               because lawn grass is background and a broadleaf spray does not kill grasses.
  grassweeds   LibreYOLO/grass-weeds (YOLO, class "ridderzuring" = broad-leaved dock in grassland) -> weed.
  (cropsweeds  toolambr/crops_weeds is NOT used: review 2026-11-24 found all 1,454 of its boxes are class
               "crop" on pictures of field weeds - it would teach weeds as `keep`.)
  openimages   Open Images V7 (FiftyOne), Flower / Houseplant / Plant / Rose / Lily boxes -> keep; images
               that show a dandelion (a weed with a flower) or only drawings are skipped.
  inat_weed    iNaturalist research-grade photos of weeds (INAT_WEED_TAXA), no boxes (labelled later).
  inat_keep    iNaturalist photos of garden plants (INAT_KEEP_TAXA), no boxes (labelled later).
  background   Open Images V7 "nothing here" ground photos (gravel, soil, lawn ... without any plant tag), no boxes.

An Item is one image: where to get it (`src`, a local path or URL), its split and its YOLO boxes
(class, cx, cy, w, h, all relative 0..1).
"""
import glob
import hashlib
import json
import os
from collections import namedtuple

WEED, KEEP = 0, 1
CLASS_NAMES = ['weed', 'keep']
GRASS_TYPE = {'goosegrass', 'crabgrass', 'barnyardgrass', 'johnsongrass', 'foxtail', 'nutsedge'}
OI_KEEP = {'Flower', 'Houseplant', 'Plant', 'Rose', 'Lily'}
OI_WEED_HINTS = {'Dandelion'}
# Image-level labels (full Open Images vocabulary, human-verified) of weeds: a picture showing any of these
# never becomes a `keep` example. FiftyOne only loads the boxable labels, so these come from the separate
# `*-annotations-human-imagelabels.csv` files, see oi_weed_image_ids().
OI_WEED_MIDS = {'/m/0fr19': 'Dandelion', '/m/02qpvtx': 'Common Dandelion', '/m/0c2pc': 'Clover',
                '/m/0grfz': 'Spurge', '/m/06tzj3': 'Thistle', '/m/08486': 'Weed', '/m/03218s': 'Dock',
                '/m/0dmw0': 'Plantago', '/m/03715w': 'Wood sorrel', '/m/03pjsn': 'Knotweed',
                '/m/03gtp5': "Lamb's quarters", '/m/0qqk': 'Amaranth'}
OI_IMAGELABEL_URLS = ['https://storage.googleapis.com/openimages/v5/validation-annotations-human-imagelabels.csv',
                      'https://storage.googleapis.com/openimages/v5/test-annotations-human-imagelabels.csv']
OI_PEOPLE = {'Person', 'Man', 'Woman', 'Boy', 'Girl', 'Human body', 'Human face'}
OI_MIN_AREA = 0.005       # keep boxes smaller than 0.5 % of the image are too small to teach anything

LICENCES = {
    'cottonweed': 'CC BY-NC 4.0 - CottonWeedDet12, Lu et al., Zenodo 10.5281/zenodo.7535814 (HF Voxel51/CottonWeedDet12)',
    'grassweeds': 'CC BY 4.0 - Roboflow 100 "grass-weeds" (HF LibreYOLO/grass-weeds)',
    'cropsweeds': 'MIT - CropWeeds-YOLO (Kaggle swish9/weeds-detection) via HF toolambr/crops_weeds',
    'openimages': 'images CC BY 2.0, boxes CC BY 4.0 - Open Images V7 (Google)',
    'background': 'images CC BY 2.0 - Open Images V7 (Google), no boxes',
}

# iNaturalist: name -> (taxon id, photos wanted). Ids looked up 2026-11-25; 1290817 / 1087592 are complexes.
INAT_WEED_TAXA = {
    'Euphorbia maculata': (58893, 3000), 'Euphorbia prostrata': (200053, 900), 'Portulaca oleracea': (58991, 735),
    'Taraxacum officinale': (47602, 735), 'Trifolium repens': (55745, 735), 'Plantago major': (58961, 735),
    'Plantago lanceolata': (53178, 735), 'Oxalis corniculata': (53168, 735), 'Polygonum aviculare': (1290817, 735),
    'Stellaria media': (1087592, 735), 'Lamium amplexicaule': (55459, 735), 'Medicago lupulina': (57056, 735),
    'Mollugo verticillata': (78090, 735)}
INAT_KEEP_TAXA = {
    'Rosa': (53438, 460), 'Lavandula': (61905, 460), 'Hydrangea macrophylla': (122034, 460),
    'Buxus sempervirens': (133397, 460), 'Tagetes': (72389, 460), 'Petunia': (55406, 460),
    'Pelargonium': (72281, 460), 'Hibiscus \u00d7 rosa-sinensis': (1628098, 460), 'Salvia': (50644, 460),
    'Hemerocallis': (82826, 460), 'Hosta': (82095, 460), 'Juniperus': (47574, 460),
    'Rhododendron': (49487, 460)}
INAT_LICENCES = {'cc0': 'CC0', 'cc-by': 'CC BY', 'cc-by-nc': 'CC BY-NC'}   # the only photo licences used

# Open Images tags of bare ground (the "nothing here" photos) and of anything plant-like (those photos are dropped).
BG_MIDS = {'/m/016b3m': 'Gravel', '/m/01tp0c': 'Pebble', '/m/01cbzq': 'Rock', '/m/09dv7': 'Soil',
           '/m/027t7q': 'Mulch', '/m/01v327': 'Lawn', '/m/08t9c_': 'Grass', '/m/02t2zd': 'Cobblestone',
           '/m/01jm2n': 'Walkway', '/m/01mxf': 'Concrete'}
PLANT_MIDS = {'/m/05s2s': 'Plant', '/m/0c9ph5': 'Flower', '/m/03fp41': 'Houseplant', '/m/07j7r': 'Tree',
              '/m/04sjm': 'Flowering plant', '/m/03gq1': 'Herb', '/m/018ssc': 'Groundcover', '/m/09t49': 'Leaf',
              '/m/0fm3zh': 'Flowerpot', '/m/0bl0l': 'Garden', '/m/0184yy': 'Moss', '/m/01xxn0': 'Hedge',
              '/m/01fnns': 'Vegetation', **OI_WEED_MIDS}

Item = namedtuple('Item', 'source name src split boxes licence orig')
SPLIT_DIRS = {'train': 'train', 'valid': 'val', 'val': 'val', 'test': 'test'}


def fo_box_to_yolo(x, y, w, h):
    """FiftyOne / Open Images box [x, y, w, h] (relative, top-left) -> YOLO (cx, cy, w, h), clamped to the image."""
    x0, y0 = max(0.0, x), max(0.0, y)
    x1, y1 = min(1.0, x + w), min(1.0, y + h)
    return ((x0 + x1) / 2.0, (y0 + y1) / 2.0, x1 - x0, y1 - y0)


def stable_split(name, seed='weedbot'):
    """train / val / test (80 / 10 / 10) from a hash of the file name: the same image always lands in the same split."""
    h = int(hashlib.sha1(f'{seed}:{name}'.encode()).hexdigest()[:8], 16) % 100
    return 'train' if h < 80 else ('val' if h < 90 else 'test')


def cottonweed_items(samples_json, image_root):
    """CottonWeedDet12 samples.json -> Items. `image_root` is a folder or the HF base URL that holds data/*.jpg."""
    with open(samples_json) as f:
        samples = json.load(f)['samples']
    items = []
    for s in samples:
        name = os.path.basename(s['filepath'])
        boxes, orig = [], []
        for det in ((s.get('ground_truth') or {}).get('detections') or []):
            orig.append(det['label'])
            if det['label'].lower() in GRASS_TYPE:
                continue
            boxes.append((WEED,) + fo_box_to_yolo(*det['bounding_box']))
        src = image_root.rstrip('/') + '/' + s['filepath']
        # split by capture session (date_device_initials): photos of one session look alike, so a whole
        # session goes to one split; otherwise near-duplicates in train and test make the test score too good
        session = '_'.join(os.path.splitext(name)[0].split('_')[:3])
        items.append(Item('cottonweed', name, src, stable_split(session), boxes, LICENCES['cottonweed'], ';'.join(orig)))
    return items


def _yolo_line_to_box(parts):
    """'cls cx cy w h' or a polygon 'cls x1 y1 x2 y2 ...' -> (cx, cy, w, h)."""
    vals = [float(v) for v in parts[1:]]
    if len(vals) == 4:
        return tuple(vals)
    xs, ys = vals[0::2], vals[1::2]
    x0, x1, y0, y1 = min(xs), max(xs), min(ys), max(ys)
    return ((x0 + x1) / 2.0, (y0 + y1) / 2.0, x1 - x0, y1 - y0)


def yolo_dir_items(root, source, names, class_map, licence):
    """A YOLO dataset folder (<split>/images/... or images/<split>/...) -> Items.
    `names` = the source's class names by id; `class_map` = name -> WEED / KEEP / None (None drops the box)."""
    root = str(root)
    items = []
    for img in sorted(glob.glob(os.path.join(root, '**', '*'), recursive=True)):
        if not img.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        rel = os.path.relpath(img, root).split(os.sep)
        if 'images' not in rel:
            continue
        split = next((SPLIT_DIRS[p] for p in rel if p in SPLIT_DIRS), None) or stable_split(rel[-1])
        lab_parts = list(rel)
        lab_parts[lab_parts.index('images')] = 'labels'
        lab = os.path.splitext(os.path.join(root, *lab_parts))[0] + '.txt'
        boxes, orig = [], []
        if os.path.exists(lab):
            with open(lab) as f:
                for line in f:
                    parts = line.split()
                    if len(parts) < 5:
                        continue
                    cname = names[int(float(parts[0]))]
                    orig.append(cname)
                    cls = class_map.get(cname)
                    if cls is not None:
                        boxes.append((cls,) + _yolo_line_to_box(parts))
        items.append(Item(source, rel[-1], img, split, boxes, licence, ';'.join(orig)))
    return items


def oi_weed_image_ids(csv_paths):
    """Open Images image ids that a human tagged with a weed label (OI_WEED_MIDS, confidence 1)."""
    import csv
    ids = set()
    for path in csv_paths:
        with open(path) as f:
            for r in csv.DictReader(f):
                if r['LabelName'] in OI_WEED_MIDS and r['Confidence'].startswith('1'):
                    ids.add(r['ImageID'])
    return ids


def openimages_items(dataset, max_images=3000, det_field='detections', exclude_ids=frozenset()):
    """FiftyOne Open Images samples -> Items with KEEP boxes only. Skips drawings (IsDepiction), images
    tagged with a weed (dandelion) or with people, images whose keep box covers a GROUP of plants
    (IsGroupOf), tiny keep boxes, and images left without any keep box. FiftyOne's Open Images zoo stores the
    boxes in the field `detections` (image-level labels in `positive_labels`)."""
    items = []
    for s in dataset:
        positives = set()
        if s.has_field('positive_labels') and s['positive_labels'] is not None:
            positives = {c.label for c in s['positive_labels'].classifications}
        if positives & (OI_WEED_HINTS | OI_PEOPLE):
            continue                                    # a weed flower in view, or a people scene
        if os.path.splitext(os.path.basename(s.filepath))[0] in exclude_ids:
            continue                                    # tagged with a weed (dandelion, clover, spurge...)
        dets = s[det_field].detections if s.has_field(det_field) and s[det_field] is not None else []
        if any(getattr(d, 'IsDepiction', False) for d in dets):
            continue                                    # drawings, not real plants
        keep_dets = [d for d in dets if d.label in OI_KEEP]
        if any(getattr(d, 'IsGroupOf', False) for d in keep_dets):
            continue                                    # one box over a whole hedge / bed: would teach "patch = keep"
        boxes = [(KEEP,) + fo_box_to_yolo(*d.bounding_box) for d in keep_dets
                 if d.bounding_box[2] * d.bounding_box[3] >= OI_MIN_AREA]
        if not boxes:
            continue
        name = os.path.basename(s.filepath)
        items.append(Item('openimages', name, s.filepath, stable_split(name), boxes, LICENCES['openimages'],
                          ';'.join(sorted({d.label for d in dets}))))
        if len(items) >= max_images:
            break
    return items


def inat_item(obs, source):
    """An iNaturalist API observation -> Item without boxes, or None. Only the FIRST photo is used and only
    when its licence is in INAT_LICENCES (cc0 / cc-by / cc-by-nc)."""
    photos = obs.get('photos') or []
    if not photos or photos[0].get('license_code') not in INAT_LICENCES:
        return None
    photo = photos[0]
    oid = obs['id']
    user = (obs.get('user') or {}).get('login') or 'unknown'
    licence = (f"{INAT_LICENCES[photo['license_code']]} - photo by {user} - "
               f"https://www.inaturalist.org/observations/{oid}")
    src = photo['url'].replace('/square.', '/large.')
    return Item(source, f"{oid}_{photo['id']}.jpg", src, stable_split(f'inat{oid}'), [], licence,
                (obs.get('taxon') or {}).get('name', ''))


def oi_background_rows(csv_paths):
    """Open Images image-level label files -> (image id, OI split, ';'-joined tag names) of photos that a human
    tagged (confidence 1) with a ground tag (BG_MIDS) and with no plant tag (PLANT_MIDS, confidence 1).
    The OI split ('validation' / 'test') comes from the file name."""
    import csv
    rows = []
    for path in csv_paths:
        base = os.path.basename(path)
        split = 'validation' if base.startswith('validation') else 'test'
        bg, plant = {}, set()
        with open(path) as f:
            for r in csv.DictReader(f):
                if not r['Confidence'].startswith('1'):
                    continue
                if r['LabelName'] in BG_MIDS:
                    bg.setdefault(r['ImageID'], []).append(BG_MIDS[r['LabelName']])
                if r['LabelName'] in PLANT_MIDS:
                    plant.add(r['ImageID'])
        rows += [(i, split, ';'.join(sorted(set(tags)))) for i, tags in bg.items() if i not in plant]
    return rows


def background_items(rows, max_images=2000):
    """oi_background_rows -> Items without boxes, sorted by image id (stable), at most max_images."""
    items = []
    for image_id, split, tags in sorted(rows)[:max_images]:
        src = f'https://open-images-dataset.s3.amazonaws.com/{split}/{image_id}.jpg'
        items.append(Item('background', f'{image_id}.jpg', src, stable_split(image_id), [],
                          LICENCES['background'], tags))
    return items

In [ ]:
%%writefile /content/weed_detector/build.py
"""build.py - write Items (sources.py) as a YOLO dataset: resized images, label files, manifest, data.yaml.

Layout:  <out>/{train,val,test}/{images,labels}/<source>__<name>.jpg|.txt, <out>/manifest.csv,
<out>/data.yaml. Resumable: an image already written is not redone, and build_source() skips a source
whose `.done_<source>` marker exists (a Colab session can stop and be re-run).
"""
import csv
import io
import os
import re
import time
import urllib.error
import urllib.request
from concurrent.futures import ThreadPoolExecutor

from PIL import Image, ImageOps

from .sources import CLASS_NAMES

MANIFEST_FIELDS = ['file', 'split', 'source', 'n_boxes', 'orig_labels', 'licence']
MAX_FAILED_SHARE = 0.005     # a source counts as finished when at most 0.5 % of its images failed to download


def _retry_wait(err, attempt):
    """Seconds to wait before the next try: the server's Retry-After for 429 / 503, else 2, 4, 8 ... (max 60)."""
    if isinstance(err, urllib.error.HTTPError) and err.code in (429, 503):
        hdrs = err.headers or {}
        try:
            return min(120.0, float(hdrs.get('Retry-After', '')))
        except (TypeError, ValueError):
            m = re.search(r't=(\d+)', hdrs.get('RateLimit', '') or '')      # Hugging Face: RateLimit: ...;t=N
            return min(120.0, float(m.group(1)) + 1.0) if m else min(60.0, 5.0 * 2 ** attempt)
    return min(60.0, 2.0 * 2 ** attempt)


def fetch(src, retries=8, sleep=time.sleep):
    """Bytes of a local file or a URL. Retries with growing waits: Hugging Face answers 429 ("too many
    requests") to anonymous downloads in bursts, and big downloads sometimes drop."""
    if not src.startswith(('http://', 'https://')):
        with open(src, 'rb') as f:
            return f.read()
    for attempt in range(retries):
        try:
            req = urllib.request.Request(src, headers={'User-Agent': 'weedbot-dataset-builder'})
            with urllib.request.urlopen(req, timeout=120) as r:
                return r.read()
        except Exception as e:
            if attempt == retries - 1 or (isinstance(e, urllib.error.HTTPError) and e.code in (400, 401, 403, 404, 410)):
                raise                                   # missing / forbidden: retrying cannot help
            sleep(_retry_wait(e, attempt))


def _out_paths(out_dir, item):
    stem = f'{item.source}__{os.path.splitext(item.name)[0]}'
    img = os.path.join(out_dir, item.split, 'images', stem + '.jpg')
    lab = os.path.join(out_dir, item.split, 'labels', stem + '.txt')
    return img, lab


def _write_one(item, out_dir, long_side, quality):
    img_path, lab_path = _out_paths(out_dir, item)
    if os.path.exists(img_path) and os.path.exists(lab_path):
        return None
    with Image.open(io.BytesIO(fetch(item.src))) as im:
        im = ImageOps.exif_transpose(im).convert('RGB')
        scale = long_side / float(max(im.size))
        if scale < 1.0:
            im = im.resize((round(im.width * scale), round(im.height * scale)), Image.LANCZOS)
        os.makedirs(os.path.dirname(img_path), exist_ok=True)
        os.makedirs(os.path.dirname(lab_path), exist_ok=True)
        im.save(img_path + '.part', 'JPEG', quality=quality)
    with open(lab_path + '.part', 'w') as f:
        for c, cx, cy, w, h in item.boxes:
            f.write(f'{c} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n')
    os.replace(lab_path + '.part', lab_path)
    os.replace(img_path + '.part', img_path)          # the image last: it marks the pair as complete
    return {'file': os.path.relpath(img_path, out_dir), 'split': item.split, 'source': item.source,
            'n_boxes': len(item.boxes), 'orig_labels': item.orig, 'licence': item.licence}


def build(items, out_dir, long_side=1024, quality=90, workers=1, log_every=500):
    """Write every item not yet written. Returns {'written': n, 'skipped_existing': n, 'failed': n}."""
    out_dir = str(out_dir)
    os.makedirs(out_dir, exist_ok=True)
    manifest = os.path.join(out_dir, 'manifest.csv')
    new_file = not os.path.exists(manifest)
    counts = {'written': 0, 'skipped_existing': 0, 'failed': 0}
    with open(manifest, 'a', newline='') as mf:
        writer = csv.DictWriter(mf, fieldnames=MANIFEST_FIELDS)
        if new_file:
            writer.writeheader()

        def job(it):
            try:
                return _write_one(it, out_dir, long_side, quality)
            except Exception as e:                      # one bad download must not stop 10,000 images
                return {'error': f'{it.source}/{it.name}: {e}'}

        with ThreadPoolExecutor(max_workers=workers) as pool:
            for i, row in enumerate(pool.map(job, items), 1):
                if row is None:
                    counts['skipped_existing'] += 1
                elif 'error' in row:
                    counts['failed'] += 1
                    print('FAILED', row['error'])
                else:
                    writer.writerow(row)
                    counts['written'] += 1
                if log_every and i % log_every == 0:
                    mf.flush()
                    print(f'{i}/{len(items)} {counts}', flush=True)
    return counts


def build_source(source, items, out_dir, **kw):
    """build() one source and leave a `.done_<source>` marker; a finished source is skipped next time."""
    marker = os.path.join(str(out_dir), f'.done_{source}')
    if os.path.exists(marker):
        return {'skipped': True}
    counts = build(items, out_dir, **kw)
    if counts['failed'] <= MAX_FAILED_SHARE * max(1, len(items)):
        with open(marker, 'w') as f:
            f.write(str(counts))
    return counts


def write_yaml(out_dir):
    """data.yaml for Ultralytics (paths relative to `path`)."""
    p = os.path.join(str(out_dir), 'data.yaml')
    names = '\n'.join(f'  {i}: {n}' for i, n in enumerate(CLASS_NAMES))
    with open(p, 'w') as f:
        f.write(f'path: {os.path.abspath(str(out_dir))}\ntrain: train/images\nval: val/images\ntest: test/images\n'
                f'names:\n{names}\n')
    from pathlib import Path
    return Path(p)

In [ ]:
%%writefile /content/weed_detector/teacher.py
"""teacher.py - OWLv2 (open-vocabulary detector) draws the boxes on images whose class is already known.

Weed photos -> `weed` boxes, garden-plant photos -> `keep` boxes, "nothing here" ground photos -> must show
no plant. Each box's class comes from the folder (kind), not from the model; OWLv2 only says WHERE a plant is.
Distractor queries (grass, stone, soil ...) compete with the plant queries so those things do not become boxes.

The raw detections (all queries, down to RAW_MIN_SCORE) go into a JSONL cache, so thresholds and the box
rules can change later without running the model again. Heavy imports (torch, transformers) stay inside
Owlv2Teacher so the unit tests run without them.
"""
import json
import os
import time
from collections import Counter, namedtuple

from . import sources as S

TEACHER_MODEL = 'google/owlv2-base-patch16-ensemble'
# Prompts and thresholds set by the VM sample check 2026-11-25 (teacher_check.py on 39 iNaturalist weed, 39 garden-plant
# and 30 Open Images ground photos; numbers in the v2 design doc, "Teacher settings from the sample check").
QUERIES = {
    # 'a plant with small leaves' -> 'a plant with reddish stems and small leaves': the old query drew whole-picture
    # boxes on any foliage (18 of the 22 'close-up' drops) and the one wrong kept weed box (on a black pot); the new
    # one is the strongest query on the spurge photos (spurge = the yard's main weed). Distractors unchanged: extra
    # grass distractors ('a blade of grass', 'a tuft of grass') stole the narrow-leaved plantain rosettes.
    'weed': {'plant': ['a weed', 'a small plant', 'a plant with reddish stems and small leaves'],
             'distract': ['grass', 'a stone', 'soil', 'a human hand', 'a shoe', 'a flower', 'a shrub']},
    # unchanged: 'a flower' stays a PLANT query (keep boxes are mostly blossoms / flower clusters). Moving it to the
    # distractors to get whole-plant boxes gave worse boxes (part of a bush, a pot with leaves) and fewer photos.
    'keep': {'plant': ['a flower', 'a shrub', 'a flowering plant', 'a bush'],
             'distract': ['grass', 'a stone', 'soil', 'a human hand', 'a shoe', 'a weed', 'a flower pot']},
    # + 'trees', 'vegetation': 4 of 23 kept ground photos showed trees / a scrub-covered hillside; these two queries
    # drop the forest, scrub and cliff-top vegetation photos (lawn grass stays below the threshold).
    'background': {'plant': ['a weed', 'a small plant', 'a flower', 'a shrub', 'trees', 'vegetation'],
                   'distract': ['grass', 'a stone', 'soil', 'gravel']},
}
# weed 0.15 unchanged: of the 15 kept weed boxes scoring 0.15-0.20 only one was loose; raising it loses real plants.
# keep 0.15 -> 0.25: the boxes scoring 0.15-0.24 were extra blossoms or wrong (stems + a person's legs); every kept
# keep photo still has its main box (0.28-0.74). background 0.12 -> 0.15: needed with 'vegetation', which scores
# 0.12-0.14 on bare rock / soil photos; every photo dropped at 0.12 before is still dropped.
THRESHOLDS = {'weed': 0.15, 'keep': 0.25, 'background': 0.15}
NMS_IOU = 0.5
MIN_BOX_AREA = 0.003     # share of the image; smaller boxes teach nothing
MAX_BOX_AREA = 0.6       # a plant filling more than this is a close-up, not a plant in a scene
RAW_MIN_SCORE = 0.05     # detections are cached down to this score
CLASS_OF = {'weed': S.WEED, 'keep': S.KEEP}

Det = namedtuple('Det', 'query score x0 y0 x1 y1')

# The prompts and thresholds above were tuned with this transformers version. 5.x pads each photo black instead of
# grey before the model sees it, which changes some keep / drop verdicts (a weed photo kept with 9 boxes on 4.57.6
# was dropped as a close-up on 5.20.2), so the teacher refuses to run on any other version.
TEACHER_TRANSFORMERS = '4.57.6'
PIN_MESSAGE = (f'transformers {TEACHER_TRANSFORMERS} is not active: click Runtime > Restart session, then '
               'Runtime > Run all (if it stops here again, tell Claude)')


def require_transformers():
    """Return TEACHER_TRANSFORMERS when it is both installed (package metadata read from disk now, so a pip
    install earlier in the session counts) and the version of any transformers module already imported in this
    Python (an older import of another version would be the one that runs). Otherwise raise RuntimeError(PIN_MESSAGE)."""
    import importlib
    import importlib.metadata
    import sys
    importlib.invalidate_caches()                    # forget cached package listings from before a pip install
    # Python 3.10's invalidate_caches() leaves importlib.metadata's own listing cache, which is keyed by the
    # folder's modification time; a pip install within one clock tick of the last check kept the old answer.
    # (3.10 defines it without @classmethod, so call it on an instance; that also works on 3.12.)
    finder = getattr(importlib.metadata, 'MetadataPathFinder', None)
    if finder is not None and hasattr(finder, 'invalidate_caches'):
        finder().invalidate_caches()
    try:
        installed = importlib.metadata.version('transformers')
    except importlib.metadata.PackageNotFoundError:
        installed = None
    mod = sys.modules.get('transformers')
    loaded = getattr(mod, '__version__', None) if mod is not None else TEACHER_TRANSFORMERS
    if installed != TEACHER_TRANSFORMERS or loaded != TEACHER_TRANSFORMERS:
        raise RuntimeError(f'{PIN_MESSAGE}. Found: installed {installed}, already imported '
                           f'{loaded if mod is not None else "none"}')
    return installed


def queries_for(kind):
    q = QUERIES[kind]
    return list(q['plant']) + list(q['distract'])


def pixel_box_to_rel(box_px, width, height):
    x0, y0, x1, y1 = box_px
    clip = lambda v: min(1.0, max(0.0, v))      # noqa: E731
    return (clip(x0 / width), clip(y0 / height), clip(x1 / width), clip(y1 / height))


def _iou(a, b):
    iw = min(a.x1, b.x1) - max(a.x0, b.x0)
    ih = min(a.y1, b.y1) - max(a.y0, b.y0)
    if iw <= 0 or ih <= 0:
        return 0.0
    inter = iw * ih
    union = (a.x1 - a.x0) * (a.y1 - a.y0) + (b.x1 - b.x0) * (b.y1 - b.y0) - inter
    return inter / union if union > 0 else 0.0


def select_boxes(dets, kind):
    """YOLO boxes (cls, cx, cy, w, h) from raw dets: plant queries only, score >= threshold, not tiny, NMS.
    Boxes above MAX_BOX_AREA are kept here; image_verdict() rejects the whole image as a close-up."""
    if kind not in CLASS_OF:
        return []
    plant = set(QUERIES[kind]['plant'])
    cand = [d for d in dets if d.query in plant and d.score >= THRESHOLDS[kind]
            and (d.x1 - d.x0) * (d.y1 - d.y0) >= MIN_BOX_AREA]
    cand.sort(key=lambda d: -d.score)
    kept = []
    for d in cand:
        if all(_iou(d, k) <= NMS_IOU for k in kept):
            kept.append(d)
    return [(CLASS_OF[kind], (d.x0 + d.x1) / 2, (d.y0 + d.y1) / 2, d.x1 - d.x0, d.y1 - d.y0) for d in kept]


def image_verdict(kind, boxes, dets):
    """Why an image must be dropped from the dataset, or None."""
    if kind == 'background':
        plant = set(QUERIES['background']['plant'])
        if any(d.query in plant and d.score >= THRESHOLDS['background'] for d in dets):
            return 'plant found'
        return None
    if not boxes:
        return 'teacher found no plant'
    if any(b[3] * b[4] > MAX_BOX_AREA for b in boxes):
        return 'close-up'
    return None


class Owlv2Teacher:
    def __init__(self, model_id=TEACHER_MODEL, device=None):
        import torch
        from transformers import Owlv2ForObjectDetection, Owlv2Processor
        require_transformers()                       # never build the teacher on another transformers version
        self.torch = torch
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        self.processor = Owlv2Processor.from_pretrained(model_id)
        self.model = Owlv2ForObjectDetection.from_pretrained(model_id).to(self.device).eval()

    def detect(self, image, queries, min_score=RAW_MIN_SCORE):
        """Dets for ALL queries >= min_score; each box carries its best query."""
        torch = self.torch
        w, h = image.size
        inputs = self.processor(text=[list(queries)], images=image, return_tensors='pt').to(self.device)
        with torch.no_grad():
            out = self.model(**inputs)
        side = max(w, h)          # OWLv2 pads the image to a square at the bottom / right
        # The image processor's own method: the same call and result in transformers 4.57 and 5.x (the processor's
        # wrapper was removed in 5.x). The teacher itself runs only on TEACHER_TRANSFORMERS (see require_transformers).
        res = self.processor.image_processor.post_process_object_detection(
            out, threshold=min_score, target_sizes=[(side, side)])[0]
        dets = []
        for box, score, lab in zip(res['boxes'].tolist(), res['scores'].tolist(), res['labels'].tolist()):
            dets.append(Det(queries[lab], float(score), *pixel_box_to_rel(box, w, h)))
        return dets


def _label_path(img_path):
    folder, name = os.path.split(img_path)
    root, images = os.path.split(folder)
    if images != 'images':
        raise ValueError(f'not in an images/ folder: {img_path}')
    return os.path.join(root, 'labels', os.path.splitext(name)[0] + '.txt')


def _load_cache(path, queries):
    cache = {}
    if not os.path.exists(path):
        return cache
    with open(path) as f:
        for line in f:
            try:
                row = json.loads(line)
                if row['queries'] == queries:
                    cache[row['stem']] = (row['w'], row['h'], [Det(*d) for d in row['dets']])
            except (ValueError, KeyError, TypeError):
                continue                    # a cut or damaged line: that image is simply detected again
    return cache


def label_images(image_paths, kind, teacher, cache_path, log_every=200):
    """Write label files for images in <root>/<split>/images/ (build.py layout) from the teacher's boxes;
    delete image + label when image_verdict() says so. Paths that no longer exist are skipped.
    Returns {'labelled', 'boxes', 'from_cache', 'dropped': {reason: n}}."""
    from PIL import Image, ImageOps
    queries = queries_for(kind)
    cache = _load_cache(cache_path, queries)
    res = {'labelled': 0, 'boxes': 0, 'from_cache': 0, 'dropped': Counter()}
    pending = []
    os.makedirs(os.path.dirname(os.path.abspath(cache_path)), exist_ok=True)

    def flush():
        if not pending:
            return
        need_nl = os.path.exists(cache_path) and os.path.getsize(cache_path) > 0
        if need_nl:
            with open(cache_path, 'rb') as f:
                f.seek(-1, os.SEEK_END)
                need_nl = f.read(1) != b'\n'
        with open(cache_path, 'a') as f:
            if need_nl:
                f.write('\n')
            f.write(''.join(pending))
            f.flush()
        pending.clear()

    def drop(img, lab, reason):
        for p in (img, lab):
            if os.path.exists(p):
                os.remove(p)
        res['dropped'][reason] += 1

    t0, done = time.time(), 0
    try:
        for img in image_paths:
            if not os.path.exists(img):
                continue
            lab = _label_path(img)
            stem = os.path.splitext(os.path.basename(img))[0]
            if stem in cache:
                dets = cache[stem][2]
                res['from_cache'] += 1
            else:
                try:                                        # only the photo itself may be 'unreadable'
                    with Image.open(img) as im:
                        im = ImageOps.exif_transpose(im).convert('RGB')
                        w, h = im.size
                except (OSError, ValueError, SyntaxError):
                    drop(img, lab, 'unreadable')
                    continue
                dets = teacher.detect(im, queries, RAW_MIN_SCORE)   # a model error crashes and keeps the photo
                pending.append(json.dumps({'stem': stem, 'w': w, 'h': h, 'queries': queries,
                                           'dets': [list(d) for d in dets]}) + '\n')
            boxes = select_boxes(dets, kind)
            reason = image_verdict(kind, boxes, dets)
            if reason:
                drop(img, lab, reason)
            else:
                with open(lab, 'w') as f:
                    for c, cx, cy, bw, bh in boxes:
                        f.write(f'{c} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n')
                res['labelled'] += 1
                res['boxes'] += len(boxes)
            done += 1
            if log_every and done % log_every == 0:
                flush()
                print(f'{done}/{len(image_paths)} images, {done / max(1e-9, time.time() - t0):.2f} img/s, '
                      f'{dict(res["dropped"])}', flush=True)
    finally:
        flush()                                       # labels already paid for survive a model error
    res['dropped'] = dict(res['dropped'])
    return res

In [ ]:
%%writefile /content/weed_detector/colab.py
"""colab.py - the glue the two Colab notebooks run: one dataset part per source on Google Drive, merging the
parts, and the source downloads. Kept here (not only in the notebooks) so it can be tested on the VM.

Drive layout (MyDrive/weedbot/):  dataset_parts/<source>.zip   one finished source each (a stopped
session only redoes the unfinished source);  runs/v1/   training checkpoints;  weed_detector_v1.zip  result.
"""
import collections
import glob
import os
import json
import shutil
import time
import urllib.parse
import zipfile
import zlib

from . import build as B
from . import sources as S

HF = 'https://huggingface.co/datasets'
EXPECTED_SOURCES = ('cottonweed', 'grassweeds', 'openimages')   # cropsweeds dropped (mislabelled, see sources.py)
V2_SOURCES = ('cottonweed', 'grassweeds', 'openimages', 'inat_weed', 'inat_keep', 'background')
TEACHER_SOURCES = {'inat_weed': 'weed', 'inat_keep': 'keep', 'background': 'background'}   # source -> teacher kind
OI_MAX_BOX_AREA = 0.4      # an Open Images picture with a box covering more than this is dropped at merge


def part_zip(drive_dir, source):
    return os.path.join(drive_dir, 'dataset_parts', f'{source}.zip')


def build_part(source, items_fn, work_dir, drive_dir, zip_name=None, **build_kw):
    """Build one source into work_dir/<source> and store it as dataset_parts/<zip_name>.zip on Drive
    (zip_name defaults to the source; the manifest is always named after the source).
    `items_fn()` is only called (downloads only start) if that zip does not exist yet."""
    target = part_zip(drive_dir, zip_name or source)
    if os.path.exists(target):
        print(f'{source}: already on Drive, skipped')
        return {'skipped': True}
    out = os.path.join(work_dir, source)
    items = items_fn()
    counts = B.build_source(source, items, out, **build_kw)
    if not os.path.exists(os.path.join(out, f'.done_{source}')):
        print(f'{source}: {counts["failed"]} of {len(items)} images failed (more than 0.5 %); run this cell again to retry them')
        return counts
    man = os.path.join(out, 'manifest.csv')
    if os.path.exists(man):
        os.replace(man, os.path.join(out, f'manifest_{source}.csv'))   # one manifest per part, no clash on merge
    os.makedirs(os.path.dirname(target), exist_ok=True)
    _zip_to_drive(out, os.path.join(work_dir, f'{source}_part'), target)
    print(f'{source}: {counts} -> {target}')
    return counts


def _zip_to_drive(folder, tmp_base, target):
    """Zip `folder` and put it at `target` through a .part file, so a stopped copy never leaves a half zip."""
    os.makedirs(os.path.dirname(target), exist_ok=True)
    tmp = shutil.make_archive(tmp_base, 'zip', folder)
    shutil.copy(tmp, target + '.part')
    os.replace(target + '.part', target)


def zip_ok(path):
    """True when the zip at `path` opens and every file in it reads back with the right checksum."""
    try:
        with zipfile.ZipFile(path) as z:
            return z.testzip() is None
    except (zipfile.BadZipFile, zlib.error, EOFError, OSError):
        return False


def label_part(source, work_dir, drive_dir, teacher_fn):
    """Turn dataset_parts/<source>_images.zip (photos without boxes, from step 3) into dataset_parts/<source>.zip
    with teacher boxes. `teacher_fn()` builds the teacher; it is only called when some photo is not in the
    cache dataset_parts/teacher_<source>.jsonl on Drive, so a stopped run continues where it stopped.
    A labelled zip that is already on Drive is tested first: a damaged one (Colab stopped while it was copied
    to Drive) is made again, from the cache when every photo is in it (minutes, no GPU work)."""
    from . import teacher as T
    target = part_zip(drive_dir, source)
    if os.path.exists(target):
        if zip_ok(target):
            print(f'{source}: already labelled on Drive, skipped')
            return {'skipped': True}
        print(f'{source}: {target} is damaged (Colab stopped while it was copied to Drive): '
              'labelling again from the saved teacher boxes')
        os.remove(target)
    images_zip = part_zip(drive_dir, f'{source}_images')
    if not os.path.exists(images_zip):
        raise RuntimeError(f'{images_zip} is missing: run step 3 (03_v2_collect_photos) until it prints DONE')
    out = os.path.join(work_dir, source)
    shutil.rmtree(out, ignore_errors=True)            # a fresh unpack; deleted photos come back and drop again from the cache
    try:
        with zipfile.ZipFile(images_zip) as f:
            f.extractall(out)
    except zipfile.BadZipFile:
        raise RuntimeError(f'{images_zip} is damaged: delete it in Google Drive and run step 3 (03_v2_collect_photos) again')
    cache = os.path.join(drive_dir, 'dataset_parts', f'teacher_{source}.jsonl')
    kind = TEACHER_SOURCES[source]
    paths = sorted(glob.glob(os.path.join(out, '*', 'images', '*.jpg')))
    lazy = _LazyTeacher(teacher_fn)
    res = T.label_images(paths, kind, lazy, cache)
    _rewrite_manifests(out)
    _zip_to_drive(out, os.path.join(work_dir, f'{source}_labelled'), target)
    print(f'{source}: {res} -> {target}')
    return res


class _LazyTeacher:
    """Builds the real teacher on the first detect() call."""
    def __init__(self, teacher_fn):
        self._fn, self._teacher = teacher_fn, None

    def detect(self, image, queries, min_score):
        if self._teacher is None:
            self._teacher = self._fn()
        return self._teacher.detect(image, queries, min_score)


def _how_to_make(drive_dir, source, v2):
    """What the owner runs to (re)make dataset part `source` (for merge_parts' messages). `v2`: the merge is
    version 2's, where step 2 (02_train) must not run again: it would change runs/v1, which v2 starts from."""
    if source in TEACHER_SOURCES:
        step4 = 'step 4 (04_v2_label_and_train)'
        if not os.path.exists(part_zip(drive_dir, f'{source}_images')):
            return f'run step 3 (03_v2_collect_photos) until it prints DONE, then {step4}'
        return f'run {step4}: it labels the photos from step 3'
    return ('run step 1 (01_build_dataset) until it prints DONE, then run this notebook again'
            + (' (not step 2: v1 is already trained)' if v2 else ''))


def merge_parts(drive_dir, out_dir, sources=EXPECTED_SOURCES):
    """Unzip the finished parts of `sources` (other parts on Drive, e.g. an old cropsweeds.zip, are ignored)
    into one dataset folder and write data.yaml. Returns summary(out_dir). A missing or damaged part raises
    with the notebook that makes it: step 1 for v1's parts, step 3 / 4 for v2's."""
    v2 = any(s in TEACHER_SOURCES for s in sources)
    missing = [s for s in sources if not os.path.exists(part_zip(drive_dir, s))]
    if missing:
        raise RuntimeError(f'missing dataset parts {missing}: '
                           + '; '.join(f'{s}: {_how_to_make(drive_dir, s, v2)}' for s in missing))
    os.makedirs(out_dir, exist_ok=True)
    for s in sources:
        try:
            with zipfile.ZipFile(part_zip(drive_dir, s)) as f:
                f.extractall(out_dir)
        except (zipfile.BadZipFile, zlib.error, EOFError):
            if s in TEACHER_SOURCES:
                fix = 'run step 4 (04_v2_label_and_train) again: it labels this part again from the saved teacher boxes'
            else:
                fix = 'delete it in Google Drive, then ' + _how_to_make(drive_dir, s, v2)
            raise RuntimeError(f'{part_zip(drive_dir, s)} is damaged: {fix}')
    B.write_yaml(out_dir)
    return summary(out_dir)


def _rewrite_manifests(out_dir):
    """Make every manifest_*.csv match the folder: `file` / `split` point to where each image is now, `n_boxes` is
    counted from its label file, and rows of images that are gone are removed."""
    import csv
    where = {}
    for split in ('train', 'val', 'test'):
        for img in glob.glob(os.path.join(out_dir, split, 'images', '*.jpg')):
            where[os.path.basename(img)] = split
    for man in glob.glob(os.path.join(out_dir, 'manifest_*.csv')):
        with open(man) as f:
            rows = list(csv.DictReader(f))
        if not rows:
            continue
        keep_rows = []
        for r in rows:
            base = os.path.basename(r['file'])
            if base in where:
                r['split'] = where[base]
                r['file'] = f"{where[base]}/images/{base}"
                lab = os.path.join(out_dir, where[base], 'labels', os.path.splitext(base)[0] + '.txt')
                if os.path.exists(lab):
                    with open(lab) as lf:
                        r['n_boxes'] = sum(1 for line in lf if line.strip())
                keep_rows.append(r)
        with open(man, 'w', newline='') as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            w.writeheader()
            w.writerows(keep_rows)


def drop_big_box_images(out_dir, source='openimages', max_area=OI_MAX_BOX_AREA):
    """Remove every `source` picture (image + label) that has any box covering more than max_area of it: such a
    box is a whole-picture box, and dropping just the box would leave the plant unlabelled. Manifests are
    rewritten. Returns the number of pictures removed."""
    removed = 0
    for split in ('train', 'val', 'test'):
        for lab in glob.glob(os.path.join(out_dir, split, 'labels', f'{source}__*.txt')):
            with open(lab) as f:
                areas = [float(p[3]) * float(p[4]) for p in (line.split() for line in f) if len(p) >= 5]
            if any(a > max_area for a in areas):
                os.remove(lab)
                img = os.path.join(out_dir, split, 'images', os.path.splitext(os.path.basename(lab))[0] + '.jpg')
                if os.path.exists(img):
                    os.remove(img)
                removed += 1
    _rewrite_manifests(out_dir)
    return removed


def repair_after_merge(out_dir, weed_ids=None, tmp_dir='/content/work_oi'):
    """Bring parts built by an OLDER step 1 up to date (a no-op for parts built by the current one):
    - cottonweed images move to the split of their capture session (no near-duplicates across splits);
    - openimages `keep` pictures that a human tagged with any weed in S.OI_WEED_MIDS are removed.
    `weed_ids` (Open Images ids) is downloaded when not given. Manifests are rewritten to match.
    Returns {'cottonweed_moved': n, 'openimages_removed': n}."""
    moved = removed = 0
    for split in ('train', 'val', 'test'):
        for img in glob.glob(os.path.join(out_dir, split, 'images', 'cottonweed__*.jpg')):
            stem = os.path.splitext(os.path.basename(img))[0]
            session = '_'.join(stem.split('__', 1)[1].split('_')[:3])
            target = S.stable_split(session)
            if target == split:
                continue
            for kind, ext in (('images', '.jpg'), ('labels', '.txt')):
                src = os.path.join(out_dir, split, kind, stem + ext)
                dst_dir = os.path.join(out_dir, target, kind)
                os.makedirs(dst_dir, exist_ok=True)
                if os.path.exists(src):
                    os.replace(src, os.path.join(dst_dir, stem + ext))
            moved += 1
    if weed_ids is None:
        import urllib.request
        os.makedirs(tmp_dir, exist_ok=True)
        csvs = []
        for url in S.OI_IMAGELABEL_URLS:
            path = os.path.join(tmp_dir, os.path.basename(url))
            if not os.path.exists(path):
                urllib.request.urlretrieve(url, path)
            csvs.append(path)
        weed_ids = S.oi_weed_image_ids(csvs)
    for split in ('train', 'val', 'test'):
        for img in glob.glob(os.path.join(out_dir, split, 'images', 'openimages__*.jpg')):
            stem = os.path.splitext(os.path.basename(img))[0]
            if stem.split('__', 1)[1] in weed_ids:
                os.remove(img)
                lab = os.path.join(out_dir, split, 'labels', stem + '.txt')
                if os.path.exists(lab):
                    os.remove(lab)
                removed += 1
    _rewrite_manifests(out_dir)
    return {'cottonweed_moved': moved, 'openimages_removed': removed}


def summary(out_dir):
    """{split: {source: images}} and {class name: boxes} from the label files."""
    images = collections.defaultdict(collections.Counter)
    boxes = collections.Counter()
    for split in ('train', 'val', 'test'):
        for lab in glob.glob(os.path.join(out_dir, split, 'labels', '*.txt')):
            images[split][os.path.basename(lab).split('__')[0]] += 1
            with open(lab) as f:
                for line in f:
                    if line.strip():
                        boxes[S.CLASS_NAMES[int(line.split()[0])]] += 1
    return {'images': {k: dict(v) for k, v in images.items()}, 'boxes': dict(boxes)}


# ---- source downloads (need internet; used in the Colab notebook) ----

def cottonweed_items_from_hf(tmp_dir):
    """CottonWeedDet12 labels file from Hugging Face; images are fetched one by one while building."""
    import urllib.request
    os.makedirs(tmp_dir, exist_ok=True)
    sj = os.path.join(tmp_dir, 'cottonweed_samples.json')
    if not os.path.exists(sj):
        urllib.request.urlretrieve(f'{HF}/Voxel51/CottonWeedDet12/resolve/main/samples.json', sj)
    return S.cottonweed_items(sj, f'{HF}/Voxel51/CottonWeedDet12/resolve/main')


def hf_yolo_items(repo, source, names, class_map, dest):
    """Download a (small) YOLO dataset from Hugging Face and convert it."""
    from huggingface_hub import snapshot_download
    snapshot_download(repo, repo_type='dataset', local_dir=dest)
    return S.yolo_dir_items(dest, source, names, class_map, S.LICENCES[source])


def openimages_items_from_zoo(tmp_dir, max_images=3000, max_samples_per_split=6000):
    """Open Images V7 validation + test pictures with plant boxes (FiftyOne), minus weed-tagged pictures."""
    import urllib.request
    import fiftyone.zoo as foz
    os.makedirs(tmp_dir, exist_ok=True)
    csvs = []
    for url in S.OI_IMAGELABEL_URLS:
        p = os.path.join(tmp_dir, os.path.basename(url))
        if not os.path.exists(p):
            urllib.request.urlretrieve(url, p)
        csvs.append(p)
    exclude = S.oi_weed_image_ids(csvs)
    print(f'openimages: {len(exclude)} pictures tagged with a weed are left out')
    items = []
    for split in ('validation', 'test'):
        ds = foz.load_zoo_dataset('open-images-v7', split=split, label_types=['detections', 'classifications'],
                                  classes=sorted(S.OI_KEEP), max_samples=max_samples_per_split,
                                  dataset_name=f'weedbot_oi_{split}', overwrite=True)
        items += S.openimages_items(ds, max_images=max_images - len(items), exclude_ids=exclude)
        if len(items) >= max_images:
            break
    return items


# ---- v2 sources: iNaturalist photos and Open Images background photos ----

INAT_API = 'https://api.inaturalist.org/v1/observations'
INAT_PER_PAGE = 200


def _inat_get_json(url, retries=8, sleep=time.sleep):
    """JSON of an iNaturalist API call; same retry rules as build.fetch."""
    return json.loads(B.fetch(url, retries=retries, sleep=sleep).decode('utf-8'))


def inat_observations(taxon_id, n, quality_grade='research', captive=None, get_json=None, sleep=time.sleep):
    """Up to n observations of a taxon whose first photo has an allowed licence (S.inat_item is not None).
    Pages by id (id_above) so a long walk never repeats; at most 1 request per second; stops at n, on an
    empty page, or when the API says nothing is left (total_results), so a taxon with few photos ends."""
    get_json = get_json or _inat_get_json
    base = {'taxon_id': taxon_id, 'photo_license': ','.join(S.INAT_LICENCES), 'per_page': INAT_PER_PAGE,
            'order_by': 'id', 'order': 'asc'}
    if quality_grade:
        base['quality_grade'] = quality_grade
    if captive is not None:
        base['captive'] = 'true' if captive else 'false'
    out, id_above, first = [], None, True
    while len(out) < n:
        params = dict(base)
        if id_above is not None:
            params['id_above'] = id_above
        if not first:
            sleep(1.0)
        first = False
        page = get_json(INAT_API + '?' + urllib.parse.urlencode(params, safe=','))
        results = page.get('results') or []
        if not results:
            break
        out += [o for o in results if S.inat_item(o, 'x') is not None]
        id_above = results[-1]['id']
        total = page.get('total_results')
        if total is not None and len(results) >= total:
            break
    return out[:n]


def inat_items_from_api(taxa, source, quality_grade='research', captive=None, max_per_taxon=None, **kw):
    """{name: (taxon id, wanted)} -> Items (no boxes); at most min(wanted, max_per_taxon) per taxon."""
    items = []
    sleep = kw.get('sleep', time.sleep)
    for i, (name, (taxon_id, wanted)) in enumerate(taxa.items()):
        if i:
            sleep(1.0)                                  # keep 1 request / second across taxa too
        n = wanted if max_per_taxon is None else min(wanted, max_per_taxon)
        got = [S.inat_item(o, source) for o in inat_observations(taxon_id, n, quality_grade, captive, **kw)]
        print(f'{source}: {name}: {len(got)} photos')
        items += got
    return items


def background_items_from_oi(tmp_dir, max_images=2000):
    """Open Images ground photos (no plant tag) from the image-level label files; images are fetched while building."""
    import urllib.request
    os.makedirs(tmp_dir, exist_ok=True)
    paths = []
    for url in S.OI_IMAGELABEL_URLS:
        p = os.path.join(tmp_dir, os.path.basename(url))
        if not os.path.exists(p):
            urllib.request.urlretrieve(url, p)
        paths.append(p)
    return S.background_items(S.oi_background_rows(paths), max_images)

In [ ]:
%%writefile /content/weed_detector/train.py
"""train.py - train / resume / evaluate / export the WeedBot weed-and-keep detector (Ultralytics YOLO11n).

Why resumable: free Colab sessions stop after a few hours. Checkpoints live in `project_dir` (Google Drive
in Colab); calling train() again continues from `last.pt` instead of starting over.
"""
import csv
import json
import os
from pathlib import Path

from .sources import CLASS_NAMES

TRAIN_DEFAULTS = dict(epochs=100, imgsz=640, patience=30, hsv_v=0.5, hsv_s=0.8, degrees=10.0, flipud=0.5,
                      batch=-1, workers=2, cache='disk')
# flipud: the camera looks DOWN at the ground, so upside-down is fine. hsv_v / hsv_s above the defaults:
# bright white stones and sun. cache='disk': a Colab T4 has only 2 CPU cores; decoding 1024-px JPEGs for
# every mosaic would leave the GPU waiting.


def run_finished(last_pt):
    """True when last.pt is a finished run (Ultralytics strips the optimizer and sets epoch -1 at the end).
    Ultralytics 8.4 does NOT raise on resume=True for a finished run: it silently starts a NEW training
    (on its coco8 default), so this must be checked before resuming."""
    import torch
    ck = torch.load(str(last_pt), map_location='cpu', weights_only=False)
    return ck.get('epoch', -1) < 0 or ck.get('optimizer') is None


def run_dir(project_dir, name='v1'):
    return Path(project_dir) / name


def train(data_yaml, project_dir, name='v1', base='yolo11n.pt', **overrides):
    """Train from `base`, or resume from <project_dir>/<name>/weights/last.pt. Returns the best.pt path."""
    from ultralytics import YOLO
    rd = run_dir(project_dir, name)
    last, best = rd / 'weights' / 'last.pt', rd / 'weights' / 'best.pt'
    if last.exists():
        if run_finished(last):
            print('training already finished:', best)
        else:
            YOLO(str(last)).train(resume=True)
        return best
    args = dict(TRAIN_DEFAULTS)
    args.update(overrides)
    YOLO(base).train(data=str(data_yaml), project=str(project_dir), name=name, exist_ok=True, **args)
    return best


def final_weights(project_dir, name='v1', dest_dir='/content/final'):
    """For a run that Colab stopped near its end (GPU limit): copy <run>/weights/best.pt to <dest_dir>/best.pt
    and strip the optimizer from the COPY only, so it can be evaluated and exported without a GPU while the
    run itself stays resumable. Returns (path, {'best_epoch', 'last_epoch', 'epochs'}); epochs count from 1,
    best_epoch is None for a run that already finished."""
    import shutil
    import torch
    from ultralytics.utils.torch_utils import strip_optimizer
    wdir = run_dir(project_dir, name) / 'weights'
    src, last = wdir / 'best.pt', wdir / 'last.pt'
    if not src.exists():
        raise FileNotFoundError(f'{src} not found: step 2 has not saved a model yet')
    ck = torch.load(str(src), map_location='cpu', weights_only=False)
    epochs = int((ck.get('train_args') or {}).get('epochs', 0))
    best_epoch = ck['epoch'] + 1 if ck.get('epoch', -1) >= 0 else None
    last_epoch = epochs
    if last.exists():
        lk = torch.load(str(last), map_location='cpu', weights_only=False)
        if lk.get('epoch', -1) >= 0:
            last_epoch = lk['epoch'] + 1
    dest = Path(dest_dir)
    dest.mkdir(parents=True, exist_ok=True)
    out = dest / 'best.pt'
    shutil.copy(src, out)
    strip_optimizer(str(out))
    return out, {'best_epoch': best_epoch, 'last_epoch': last_epoch, 'epochs': epochs}


def _source_lists(data_dir, split):
    """{source: [image paths]} for one split, from the `<source>__` file-name prefix."""
    out = {}
    idir = Path(data_dir) / split / 'images'
    for p in sorted(idir.glob('*.jpg')):
        out.setdefault(p.name.split('__')[0], []).append(str(p))
    return out


def evaluate_per_source(best_pt, data_dir, split='test', **val_kw):
    """mAP50 per class, precision and recall for each source's images in `split`."""
    from ultralytics import YOLO
    model = YOLO(str(best_pt))
    results = {}
    tmp = Path(data_dir) / '_eval'
    tmp.mkdir(exist_ok=True)
    for source, paths in _source_lists(data_dir, split).items():
        lst = tmp / f'{source}_{split}.txt'
        lst.write_text('\n'.join(paths) + '\n')
        yml = tmp / f'{source}_{split}.yaml'
        names = ''.join(f'  {i}: {n}\n' for i, n in enumerate(CLASS_NAMES))
        yml.write_text(f'path: {Path(data_dir).resolve()}\ntrain: {lst}\nval: {lst}\ntest: {lst}\nnames:\n{names}')
        m = model.val(data=str(yml), split='val', plots=False, verbose=False, **val_kw)
        ap50 = {CLASS_NAMES[int(c)]: round(float(v), 3) for c, v in zip(m.box.ap_class_index, m.box.ap50)}
        results[source] = {'images': len(paths), 'map50': ap50, 'precision': round(float(m.box.mp), 3),
                           'recall': round(float(m.box.mr), 3)}
    return results


def evaluate_all(best_pt, data_yaml, project_dir, split='test', **val_kw):
    """Whole test split: precision, recall and mAP50 PER CLASS (keep recall = how many of the plants to
    protect are found) plus the confusion matrix plot, saved under <project_dir>/test_eval."""
    from ultralytics import YOLO
    m = YOLO(str(best_pt)).val(data=str(data_yaml), split=split, plots=True, project=str(project_dir),
                              name='test_eval', exist_ok=True, verbose=False, **val_kw)
    per_class = {}
    for i, c in enumerate(m.box.ap_class_index):
        per_class[CLASS_NAMES[int(c)]] = {'precision': round(float(m.box.p[i]), 3), 'recall': round(float(m.box.r[i]), 3),
                                          'map50': round(float(m.box.ap50[i]), 3)}
    return {'per_class': per_class, 'map50_all': round(float(m.box.map50), 3),
            'confusion_matrix': str(Path(project_dir) / 'test_eval' / 'confusion_matrix.png')}


def export(best_pt, imgsz=640, formats=('onnx', 'ncnn')):
    """Export for the Pi: ONNX and NCNN (the Pi 4 runs NCNN fastest on its CPU). A failing format is
    reported, not raised: best.pt is what matters, and NCNN can also be made later on the VM."""
    from ultralytics import YOLO
    out = {}
    for f in formats:
        try:
            out[f] = YOLO(str(best_pt)).export(format=f, imgsz=imgsz)
        except Exception as e:
            print(f'export {f} failed: {e}')
            out[f] = None
    return out


def summary(project_dir, name='v1'):
    """Last row of results.csv (per-epoch metrics) as a dict, for a quick look."""
    p = run_dir(project_dir, name) / 'results.csv'
    if not p.exists():
        return {}
    rows = list(csv.DictReader(open(p)))
    return {k.strip(): v for k, v in rows[-1].items()} if rows else {}


if __name__ == '__main__':
    import argparse
    ap = argparse.ArgumentParser()
    ap.add_argument('data_yaml'); ap.add_argument('project_dir'); ap.add_argument('--epochs', type=int, default=100)
    a = ap.parse_args()
    b = train(a.data_yaml, a.project_dir, epochs=a.epochs)
    print(json.dumps(evaluate_per_source(b, os.path.dirname(a.data_yaml)), indent=1))

In [ ]:
!pip -q install ultralytics==8.4.218
# The teacher must be transformers 4.57.6: its prompts and thresholds were tuned with that version on the VM
# (5.x pads the photos black instead of grey, which moves the scores a little). 4.57.6 does not need torch, so
# Colab's torch stays. Nothing may import transformers before this cell.
import importlib.metadata


def version_of(pkg):
    try:
        return importlib.metadata.version(pkg)
    except importlib.metadata.PackageNotFoundError:
        return None


torch_before = version_of('torch')
if version_of('transformers') != '4.57.6':
    !pip -q install transformers==4.57.6
assert version_of('torch') == torch_before, 'pip changed torch: Runtime -> Restart session, then Run all again'
from weed_detector import teacher as TE
transformers_version = TE.require_transformers()      # STOPS here unless 4.57.6 is installed and active
print('transformers', transformers_version, '| ultralytics', version_of('ultralytics'), '| torch', torch_before)

In [ ]:
import gc, json, torch
from weed_detector import colab as C, teacher as TE
print('teacher runs on transformers', TE.require_transformers())   # stops unless 4.57.6 (the tuned teacher)


def label_all():
    """Boxes for each photo part. label_part builds the teacher only when some photo is not cached yet and drops
    it when it returns: no name holds the teacher afterwards, so there is nothing left to del."""
    return {s: C.label_part(s, '/content/work_v2', DRIVE, lambda: TE.Owlv2Teacher()) for s in C.TEACHER_SOURCES}


labelled = label_all()          # a part already labelled on Drive is skipped
gc.collect()
torch.cuda.empty_cache()        # the teacher's GPU memory goes back for the training
print(json.dumps(labelled, indent=1))

In [ ]:
from weed_detector import colab as C, train as T
DS = '/content/weeds_ds_v2'
info = C.merge_parts(DRIVE, DS, sources=C.V2_SOURCES)   # v1's three parts + the three labelled v2 parts
print('repair of parts from an older step 1:', C.repair_after_merge(DS))   # no-op for current parts
print('Open Images pictures with a box over 40 % of the picture, removed:', C.drop_big_box_images(DS))
info = C.summary(DS)
print(info)

In [ ]:
base, _ = T.final_weights(DRIVE + '/runs', 'v1', '/content/v1_final')     # v2 starts from v1's best model
best = T.train(DS + '/data.yaml', DRIVE + '/runs', name='v2', base=str(base), epochs=50, patience=15, imgsz=640)
print('best model:', best)

In [ ]:
import json
per_source = T.evaluate_per_source(best, DS, split='test', imgsz=640)
overall = T.evaluate_all(best, DS + '/data.yaml', DRIVE + '/runs/v2', split='test', imgsz=640)
print(json.dumps({'overall_test': overall, 'per_source_test': per_source}, indent=1))
json.dump({'overall_test': overall, 'per_source_test': per_source, 'dataset': info},
          open(DRIVE + '/runs/v2/metrics_per_source.json', 'w'), indent=1)

In [ ]:
import glob, shutil
exported = T.export(best, imgsz=640, formats=('onnx', 'ncnn'))
print('exported:', exported)
out = '/content/weed_detector_v2'
os.makedirs(out, exist_ok=True)
run = DRIVE + '/runs/v2'
W = run + '/weights'
for f in ['results.csv', 'confusion_matrix.png', 'results.png', 'metrics_per_source.json', 'args.yaml']:
    if os.path.exists(os.path.join(run, f)):
        shutil.copy(os.path.join(run, f), out)
for f in ['best.pt', 'best.onnx']:
    if os.path.exists(os.path.join(W, f)):
        shutil.copy(os.path.join(W, f), out)
if os.path.exists(os.path.join(run, 'test_eval', 'confusion_matrix.png')):
    shutil.copy(os.path.join(run, 'test_eval', 'confusion_matrix.png'), os.path.join(out, 'test_confusion_matrix.png'))
ncnn = os.path.join(W, 'best_ncnn_model')
if os.path.isdir(ncnn):
    shutil.copytree(ncnn, os.path.join(out, 'best_ncnn_model'), dirs_exist_ok=True)
for src in sorted(glob.glob(DS + '/manifest_*.csv')):
    shutil.copy(src, out)
open(os.path.join(out, 'README.txt'), 'w').write(
    'WeedBot weed/keep detector v2 (YOLO11n fine-tuned from v1, classes 0 weed, 1 keep, imgsz 640).\n' + '' +
    'Trained on:\n'
    '- CottonWeedDet12 (CC BY-NC 4.0, Lu et al., Zenodo 10.5281/zenodo.7535814)\n'
    '- Roboflow-100 grass-weeds (CC BY 4.0)\n'
    '- Open Images V7 (CC BY 2.0 images, CC BY 4.0 boxes): keep plants and the "nothing here" ground photos\n'
    '- iNaturalist photos: each photo has its own licence (CC0, CC BY or CC BY-NC); the observer and the URL of '
    'every photo are in manifest_inat_weed.csv / manifest_inat_keep.csv\n'
    'The boxes on the iNaturalist photos were drawn by OWLv2 (google/owlv2-base-patch16-ensemble, Apache 2.0), '
    'which also checked that the ground photos show no plant.\n'
    'Non-commercial use (CottonWeedDet12 and the CC BY-NC photos).\n')
shutil.make_archive(DRIVE + '/weed_detector_v2', 'zip', out)
size = round(os.path.getsize(DRIVE + '/weed_detector_v2.zip') / 1e6, 1)
from google.colab import drive
drive.flush_and_unmount()          # make sure the zip is really written to Google Drive
print('ALL DONE -> MyDrive/weedbot/weed_detector_v2.zip', size, 'MB')